# Day two — Biodiversity, Land Use and Habitat Change Detection
## *PART I: Philippines Climate Zones, revisited*

Yesterday, we wrapped up with the figures from Beck et al. (2023) that showed us both the current Köppen-Geiger zones and those projected under a future climate change scenario. First things first, re-run the setup cell below if the project is not loaded in your Google Colab environment yet:

In [ ]:
# 1
import os                                                                          # in English:
if not os.path.exists("/content/USLS-Workshop"):                                   # IF workshop isn't downloaded yet:
    !git clone "https://github.com/Cas-Dec/USLS-Workshop.git" /content/USLS-Workshop  # download the workshop
# open the workshop
%cd /content/USLS-Workshop
!pip install -q .                                                                  # install what is needed to do the workshop

from google.colab import drive
drive.mount('/content/drive')                                                      # connect to your Google Drive

### 1) Day two and we're expert programmers: let's build our own climate zones!

Yesterday you classified **Bacolod alone** by hand-computing four numbers and running them through an `if/elif` chain. Today we generalize that into a real, reusable *function*, finish the decision tree we left incomplete, and run it not just for one city but across the **entire Philippines**, at **three points in time**.

Three files already sit in `PROCESSED_DIR` (monthly `air_temperature` + `precipitation`, 0.1° grid, Philippines-wide):
- Past: `climate_philippines_1901_1930_mean.nc`
- Present: `climate_philippines_1991_2020_mean.nc`
- Future (SSP4-6.0): `climate_philippines_2071_2099_ssp460_mean.nc`

*(We also asked yesterday: could we get our hands on a land-sea mask to go with this? Turns out we don't need to download one — the ocean is simply `NaN` in these files, since Beck et al.'s land climate ensemble has no value over open water. That NaN pattern **is** our land-sea mask, for free.)*

See the outline below for the exercise-by-exercise plan — nothing to run yet, this is for review before we write the actual fill-in-the-blank cells.

#### PLAN — Part I exercises (for review)

**New concepts introduced today:** `def` functions, `return`, `for` loops (incl. nested loops). Neither was actually hands-on in Day 1 (only mentioned in passing) — this is their real introduction.

1. **From decision tree to function.** Wrap yesterday's `if/elif` logic into
   `def classify_koppen(coldest_month_T, warmest_month_T, annual_precip, driest_month_P): ... return climate_code`.
   Sanity-check by plugging in Bacolod's own numbers from yesterday and confirming the same result comes out.

2. **Completing the decision tree.** Yesterday we only finished Group A. Fill in:
   - The Group B (arid) threshold properly (yesterday used a placeholder `annual_precip < ...`)
   - Groups C / D / E conditions from yesterday's table (coldest month −3–18°C → C, < −3°C → D, warmest < 10°C → E)
   - The missing **Am** subtype condition — yesterday's `if/elif/else` example cell already gave us the formula (`driest < 60 and annual >= 25 * (100 - driest)`); today we actually use it.

3. **Loops warm-up.** Before looping over a whole grid: a `for` loop over a short hand-written list of `(city, T, P, ...)` tuples for a handful of PH cities, printing each city's classification. Small, ungraded, just to get `for` under their fingers.

4. **Load the gridded present-day climate data.** Open `climate_philippines_1991_2020_mean.nc`, inspect dims (`time`=12 months, `lat`=170, `lon`=135), pull out the `air_temperature` and `precipitation` arrays.

5. **Classify an entire grid.** Nested `for` loop over lat/lon indices. For each pixel: pull its 12 monthly values, compute the same 4 summary numbers as yesterday (but now per-pixel), call `classify_koppen(...)`, store the result in a pre-allocated 2D output array (`np.empty` / `np.full`, filled with NaN/placeholder where the pixel is ocean).

6. **Map it.** Reuse Day 1's `KG_INFO` colour scheme + `pcolormesh` to plot the homemade present-day map.

7. **Sanity-check against Beck et al.** Plot the homemade 0.1° map next to the official `kg_philippines_present.nc` (1 km). Discuss: do they roughly agree? Why might they differ (resolution, our simplified thresholds, missing sub-criteria)?
   *Design call: keep this comparison visual/qualitative only — computing a formal pixel-agreement score would require regridding one dataset onto the other's grid, which feels like a rabbit hole at this skill level. Flagging in case you'd rather we attempt it anyway.*

8. **Past, present, future.** Repeat steps 4–6 for `climate_philippines_1901_1930_mean.nc` and `climate_philippines_2071_2099_ssp460_mean.nc` → three maps total.

9. **Wrap-up discussion.** What changes across the three maps? Does the shift match what Beck's own past→future comparison showed yesterday? Zoom back into Bacolod specifically: does the homemade model predict Bacolod's class changing over time?

**REVIEW**
Suggested exercises look great. Let's build this. Relating to the markdown cell above: if the Beck climate files implicitly contain a land-sea mask (NaNs over ocean), then let's formally covert that into a land-sea-mask_0p1.nc in the PROCESSED_DIR.

**Open questions for you:**
1. Happy with the B/C/D/E thresholds being simplified versions of the real Köppen criteria (as in Day 1), or should we be more rigorous?
2. OK with skipping a formal accuracy/agreement score in step 7 (see design call above)?
3. Time-check: steps 1–9 feel like ~1.5–2h given two brand-new concepts (functions + loops). Does that match your budget expectations for Part I?

**ANSWERS**
1. Thresholds can be more rigorous today.
2. OK with skipping formal agreement in step 7.
3. Time-check good.

## *PART II: The Philippines as a Biodiversity Hotspot*

#### PLAN — Part II exercises (for review)

Three subsections, each building toward Part III (satellite imagery → land use) and Part IV (land use change). Data used here all comes from `PROCESSED_DIR / "day_2"`.

### 1) Vegetation types & forest cover
1. Short intro: Köppen designed his system around vegetation zones (Day 1 recap); today we look at *actual* vegetation/forest-cover data instead of a climate proxy for it.
2. Load `negros_forest_change.nc`, but only the `treecover2000` variable for now (percent canopy cover in year 2000) — `lossyear` is saved for Part IV.
3. Map `treecover2000` over Negros, reusing Day 1's map-making pattern (`pcolormesh`/`imshow`, colorbar, Bacolod marker).
4. 🖊️ Simple summary stats: mean tree cover % over Negros; % of pixels above some threshold (e.g. >50% cover) — light numpy/pandas aggregation practice.
5. 💬 Discussion: where is Negros most forested? Does that line up with the wetter (Af/Am) vs. drier (Aw) zones from Part I's homemade climate map?

### 2) Satellite imagery
1. Intro: what does a satellite actually measure? Multispectral bands vs. what our eyes see; recap the `red`/`nir`/`visual` bands sitting in `negros_sentinel2.nc`.
2. Load the true-colour bands (`visual_r/g/b`) for the "new" (~2024) time slice, stack them into a single RGB image array (new skill: `np.dstack` or `np.stack(..., axis=-1)`), display with `plt.imshow()` — first time displaying an actual photo-like image rather than a data map.
3. 🖊️ Two-panel figure: "old" (~2018) true-colour image next to "new" (~2024), reusing Day 1's subplot skills. 💬 Discussion: what visible changes can you already spot, just by eye?
4. 💬 Discussion: relate visible landscape features (forest, farmland, urban, coastline) to what you'd expect from the `treecover2000` map and from Bacolod's known location.

### 3) Fauna
1. Load `gbif_philippine_eagle.csv` and `gbif_flying_fox.csv`; scatter the occurrence points over a Philippines outline (reusing Day 1 map-making skills — plain scatter, no basemap needed).
2. 💬 Discussion: the Philippine Eagle is critically endangered and forest-dependent — relate the sparse, clustered points to forest cover (preview of Part IV's forest-loss story).
3. Introduce the richer `gbif_mammals_philippines.csv` (289 species, 48,640 records): inspect columns, `.value_counts()` on `family`/`order` to see which mammal groups dominate the record (bats vastly outnumber everything else — a nice surprise).
4. 🖊️ Filter to Negros using the `county`/`stateProvince` columns (pandas boolean-filtering practice, same pattern as Day 1's `merge`/filtering), then map those points over Negros — ties directly into the Sentinel-2/Hansen Negros maps from earlier.
5. 🖊️ Bar chart of `iucnRedListCategory` value counts (LC/NT/VU/EN/CR/DD) for the Negros subset — conservation-status storytelling.
6. 💬 Discussion: what do the different IUCN threat categories mean? Which mammal species on Negros are most at risk, and why might that connect to what we're about to do in Parts III–IV?

**REVIEW:**
- First subsection looks good. In day 1, I favoured the simple built-in plotting functionality of xarray rather than resorting to e.g. pcolormesh. We can teach this, but be gentle about it.
- Second subsection also looks good, although I'm unsure why the 'old' reference is as new as 2018. Can we go further back, to e.g. 2000, or does the sentinel database not allow this, or does it bring too much trouble for comparison?
- Third subsection looks really good.

**Open questions for you:**
- Subsection 1 introduces forest cover before Part IV needs `lossyear` — happy with splitting the Hansen dataset across two parts like this, or would you rather introduce the whole dataset (cover + loss) together in Part IV only, and cut subsection 1 down to just a climate/vegetation-zone discussion (no new data)?
- Subsection 3 step 3–4 leans on `family`/`order`/`county` columns existing and being reasonably populated in the mammal CSV — want a quick data-quality check (null rates) included as a taught step, or should we just handle it silently in the setup?
- Time-check: these three subsections feel like ~1.5–2h combined. Matches your expectations for Part II?

**ANSWERS:**
- Happy with current proposal.
- Data quality checking is good teaching material, let's include it in this notebook.
- Time check good.

## *PART III: From satellite imagery to land use*

## *PART IV: Land Use Change*

### 1) Forest cover loss
... -> using the satellite imagery from the previous exercises

### 2) Urban expansion
... -> same same